In [3]:
import torch
import torch.nn as nn
from torchvision import transforms
from PIL import Image
import warnings
import os
warnings.filterwarnings('ignore')


In [5]:
distortion_classes = ['Broken Line', 'Edge False Color', 'Over Desaturation', 'Saturated False Color', 'Smears']
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [10]:
class multi_class_model(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.features = nn.Sequential(
            # Block 1: 50x50 -> 25x25
            nn.Conv2d(3, 20, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            # Block 2: 25x25 -> 12x12
            nn.Conv2d(20, 40, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            # Block 3: 12x12 -> 6x6
            nn.Conv2d(40, 80, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            # 80 filters * 6 * 6 spatial size = 2880
            nn.Linear(80 * 6 * 6, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

# Instantiate and verify
multi_class_model = multi_class_model(len(distortion_classes)).to(device)
print(f"Total parameters: {sum(p.numel() for p in multi_class_model.parameters()):,}")

Total parameters: 406,093


In [11]:
multi_class_model.load_state_dict(torch.load('multi_class_model_weights.pth', map_location=device))

<All keys matched successfully>

In [12]:

# 3. Set to evaluation mode (disables dropout/batchnorm updates)
multi_class_model.to(device)
multi_class_model.eval()

# 4. Define the same transformation used during your validation phase
val_transform = transforms.Compose([
    transforms.Resize((50, 50)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

distortion_classes = ['Broken Line', 'Edge False Color', 'Over Desaturation', 'Saturated False Color', 'Smears']

In [13]:
def classify_distortion(patch_image):
    """Runs a distorted 50x50 PIL image patch through the 5-class model."""
    img_tensor = val_transform(patch_image).unsqueeze(0).to(device)
    with torch.no_grad():
        outputs = multi_class_model(img_tensor)
        _, pred = torch.max(outputs, 1)
    return distortion_classes[pred.item()]